# 01 — Data preprocessing

The model is downstream. Most real-world bugs are in this notebook's worth of code: bad dtype, wrong channel order, forgotten normalization, wrong image size.

## Roadmap
1. Image as a NumPy array
2. dtype and value range — the most common bug
3. Channel order: RGB vs BGR vs CHW vs HWC
4. **Image size — decision table for classifier vs detector vs regressor**
5. Normalization: why, and how to compute the constants
6. Augmentation: random flip, crop, color jitter in NumPy
7. One-hot encoding (classifier) vs scalar target (regressor)


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. Image as a NumPy array

A color image is a 3-D array: `(height, width, channels)`. Each channel value is typically 0..255 (uint8) or 0..1 (float).


In [ ]:
from PIL import Image
import numpy as np

# Make a synthetic 4×6 RGB image: red square on white background
arr = np.full((4, 6, 3), 255, dtype=np.uint8)
arr[1:3, 1:3] = [255, 0, 0]
img = Image.fromarray(arr, mode="RGB")
print("shape:", arr.shape)
print("first pixel (R,G,B):", arr[0, 0])
img.resize((180, 120))   # display upscale


## 2. dtype + value range (the most common silent bug)

| Where | Typical dtype | Typical range |
|---|---|---|
| Disk (jpg/png) | uint8 | 0..255 |
| Just-loaded NumPy | uint8 | 0..255 |
| **Model input** | **float32** | **0..1 or normalised** |
| Model output | float32 | depends on layer |

**Always convert to float and divide by 255** before feeding a model. uint8 arithmetic wraps around silently — `np.uint8(200) + np.uint8(100) == 44`.


In [ ]:
u = np.array([200, 100], dtype=np.uint8)
print("uint8 add:", u.sum())               # 44 — wraparound!
f = u.astype(np.float32) / 255.0
print("float add:", f.sum())               # 1.176 — correct


## 3. Channel order: RGB vs BGR, HWC vs CHW

- **PIL / matplotlib / torchvision** use **HWC + RGB**.
- **OpenCV** uses **HWC + BGR**.
- **PyTorch tensors** use **CHW + RGB**.

Mixing these is the #2 most common bug — model trained with one order tested with another quietly drops 20+ accuracy points.


In [ ]:
# Convert HWC → CHW (NumPy axis transpose)
hwc = np.zeros((32, 32, 3), dtype=np.float32)
chw = hwc.transpose(2, 0, 1)                 # axes: 2→0, 0→1, 1→2
print("HWC:", hwc.shape, " → CHW:", chw.shape)


## 4. Image-size decision rules

You **never** feed the raw image. You pick a target size based on the *task*:

| Task | Standard input size | Why |
|---|---|---|
| Image classifier (ResNet-style) | **224 × 224** | Pretrained ImageNet weights use this. Smaller loses fine detail; larger costs 4x memory per doubling. |
| Object detector (YOLO-style) | **640 × 640** | Bigger so small objects survive. Must be divisible by 32 (the network's downsampling stride). |
| Pixel-precise (segmentation) | 512–1024 | Need spatial detail in the output. |
| Image regressor (cost, age, …) | 224 × 224 | Same as classifier; the head is a single scalar instead of N classes. |

**Output-size formula for a conv layer:**

$$H_\text{out} = \left\lfloor \frac{H_\text{in} + 2P - K}{S} \right\rfloor + 1$$

where $K$ is kernel size, $S$ is stride, $P$ is padding. Worked example: a 224×224 image through a 7×7 conv with stride 2 and padding 3:

$$\left\lfloor \frac{224 + 6 - 7}{2} \right\rfloor + 1 = 112$$


In [ ]:
def conv_output_size(H_in, K, S, P):
    return (H_in + 2*P - K) // S + 1

print("first conv of ResNet50:", conv_output_size(224, K=7, S=2, P=3))   # 112
print("after maxpool 3×3 s2 p1:", conv_output_size(112, K=3, S=2, P=1))  # 56


## 5. Normalization — why and how

Networks train better when inputs are roughly mean 0, std 1. Two reasons:

1. **Activations stay in the useful range** of sigmoid / tanh / softmax.
2. **Gradient magnitudes are stable** across layers — avoids vanishing/exploding gradients.

For ImageNet-pretrained models, the standard constants are computed once over millions of training images:

```
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

For your own dataset, compute them from the train split.


In [ ]:
# Compute CIFAR-10-style normalization constants from a small batch.
imgs = np.random.rand(100, 32, 32, 3).astype(np.float32)   # fake
mean = imgs.mean(axis=(0, 1, 2))
std  = imgs.std(axis=(0, 1, 2))
print("mean per channel:", mean.round(3), "  std:", std.round(3))


## 6. Augmentation — three useful ones in 10 lines of NumPy


In [ ]:
def random_horizontal_flip(img, p=0.5):
    return img[:, ::-1] if np.random.rand() < p else img

def random_crop_and_resize(img, crop_size, out_size):
    """Crop a random crop_size×crop_size region then upscale (nearest) to out_size."""
    H, W = img.shape[:2]
    y = np.random.randint(0, H - crop_size + 1)
    x = np.random.randint(0, W - crop_size + 1)
    crop = img[y:y+crop_size, x:x+crop_size]
    # Nearest-neighbour upscale; for production prefer LANCZOS via PIL.
    factor = out_size / crop_size
    rows = (np.arange(out_size) / factor).astype(int)
    cols = (np.arange(out_size) / factor).astype(int)
    return crop[np.ix_(rows, cols)]

def color_jitter(img, strength=0.1):
    return np.clip(img + np.random.uniform(-strength, strength, size=3), 0, 1)

# Quick demo on a synthetic image
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
src = np.random.rand(32, 32, 3).astype(np.float32)
for ax, fn, t in zip(axes,
                      [lambda x: x, random_horizontal_flip,
                       lambda x: random_crop_and_resize(x, 24, 32),
                       color_jitter],
                      ["original", "h-flip", "random crop+resize", "color jitter"]):
    ax.imshow(fn(src)); ax.set_title(t); ax.axis("off")
plt.show()


## 7. Targets: classifier vs regressor

The model's *output head* and the *loss function* depend on what you want it to predict.

| Task | Target shape | Final layer | Loss |
|---|---|---|---|
| Multi-class (one of N) | scalar int $\in [0, N)$ | N logits | Softmax + cross-entropy |
| Multi-label (any subset of N) | length-N vector of 0/1 | N logits | N independent BCEs |
| Regression (scalar) | one float | 1 linear output | MSE or MAE |
| Detection (boxes + classes) | list of (cls, bbox) | per-grid-cell predictions | sum of regression + classification |


In [ ]:
# One-hot encode an integer label
def one_hot(y, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    out[y] = 1.0
    return out

print(one_hot(3, num_classes=10))   # plane=0 ... dog=5 ... so index 3 = cat → [0,0,0,1,0,0,0,0,0,0]


**Next:** build a single neuron, then a 2-layer MLP, and train it on a toy problem.
